# WaveForge 3D — CPU vs GPU Benchmark
### PyMEEP (CPU) · WaveForge CPU · WaveForge GPU (T4 / A100)

This notebook runs on **Google Colab** with a T4 or A100 GPU.
It benchmarks all 10 3D examples and compares:
- PyMEEP on CPU (reference)
- WaveForge on CPU
- WaveForge on GPU (this Colab session)

**Runtime: GPU** (Runtime → Change runtime type → GPU)

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import subprocess, sys, os

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'NOT FOUND — switch to GPU runtime!')

# Clone the repo
if not os.path.exists('GPU-MEEP'):
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/GPU-MEEP.git'], check=True)
os.chdir('GPU-MEEP')
sys.path.insert(0, 'src')
print('Repo ready. Torch:', __import__('torch').__version__)

In [ ]:
# ── 2. Verify CUDA available ───────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'No CUDA — change runtime to GPU!'
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Install PyMEEP for reference baseline ──────────────────────────────────
# (skip if you just want WaveForge GPU vs CPU)
INSTALL_MEEP = False  # set True to install PyMEEP (~5 min)
if INSTALL_MEEP:
    subprocess.run(['conda', 'install', '-c', 'conda-forge', 'meep', '-y'], check=True)
    print('PyMEEP installed')
else:
    print('Skipping PyMEEP install — will use pre-recorded CPU baseline from repo')

In [ ]:
# ── 4. Load pre-recorded baselines from repo ──────────────────────────────────
import json
import numpy as np

# CPU results (recorded locally on Kali dev machine)
with open('benchmarks/3d_examples_cpu_results.json') as f:
    cpu_3d = json.load(f)

# Kaggle T4 results (recorded on Kaggle 2xT4)
with open('benchmarks/kaggle_gpu_results.json') as f:
    kaggle = json.load(f)

# PyMEEP vs WaveForge CPU (2D)
with open('benchmarks/meep_comparison_results.json') as f:
    meep_cmp = json.load(f)

# Scaling (CPU 2D)
with open('benchmarks/cpu_results.json') as f:
    cpu_scaling = json.load(f)

print('Baselines loaded:')
print(f'  3D CPU examples:   {len(cpu_3d)} entries')
print(f'  Kaggle T4 scaling: {len(kaggle["gpu_scaling"]["cuda:0"]["rows"])} grid sizes')
print(f'  PyMEEP comparison: {len(meep_cmp)} scenarios')

In [ ]:
# ── 5. Run WaveForge GPU benchmark (grid scaling) ────────────────────────────
import time, math
from core.grid import YeeGrid
from core.fields import FieldSet
from core.boundaries import MurABC3D
from core.sources import GaussianPulse, PointSource, SourceCollection
from core.fdtd3d import FDTD3D

DEVICE = 'cuda'
N_WARMUP = 20
N_STEPS  = 100
GRID_SIZES = [32, 48, 64, 96, 128, 192, 256]

def benchmark_grid(N, device='cuda', n_warmup=20, n_steps=100):
    DX = 1.5e-3
    grid = YeeGrid(N, N, dx=DX, dy=DX, Nz=N, dz=DX, device=device)
    fields = FieldSet(grid)
    boundary = MurABC3D(grid, fields.Hx, fields.Hy, fields.Hz)
    pulse = GaussianPulse(amplitude=1.0, sigma=20*grid.dt)
    cx = cy = cz = N // 2
    src = PointSource(pulse, cx, cy, 'Ez', k=cz, grid=grid, N_steps=n_warmup+n_steps)
    sim = FDTD3D(grid, fields, boundary, SourceCollection([src]), n_check=99999)

    # Warmup
    with torch.no_grad():
        sim.run(n_warmup)
    if device == 'cuda': torch.cuda.synchronize()

    # Timed run
    t0 = time.perf_counter()
    with torch.no_grad():
        sim.run(n_steps)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    total_cells = n_steps * N**3
    mcells_s = total_cells / elapsed / 1e6
    ms_step = elapsed / n_steps * 1e3
    return mcells_s, ms_step

print('Running GPU grid scaling benchmark...')
gpu_scaling_results = []
for N in GRID_SIZES:
    try:
        mcells, ms = benchmark_grid(N, 'cuda', N_WARMUP, N_STEPS)
        gpu_scaling_results.append({'N': N, 'mcells_s': round(mcells,1), 'ms_step': round(ms,3)})
        print(f'  {N:4d}³: {mcells:7.1f} Mcells/s  ({ms:.3f} ms/step)')
    except torch.cuda.OutOfMemoryError:
        print(f'  {N:4d}³: OOM')
        break

print('Done.')

In [ ]:
# ── 6. Run all 10 3D examples on GPU ─────────────────────────────────────────
import subprocess, re, time

PYTHON = sys.executable
gpu_example_results = []

for entry in cpu_3d:
    fname = entry['file']
    path = f'examples/3d/{fname}'
    print(f'Running {fname}...', end=' ', flush=True)
    t0 = time.time()
    try:
        result = subprocess.run([PYTHON, path], capture_output=True, text=True, timeout=300,
                                env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
        elapsed = time.time() - t0
        output = result.stdout + result.stderr
        m = re.search(r'WAVEFORGE_BENCH:\s*([\d.]+)', output)
        g = re.search(r'Grid:\s*(\d+)x(\d+)x(\d+)', output)
        mcells = float(m.group(1)) if m else 0
        grid_str = f'{g.group(1)}x{g.group(2)}x{g.group(3)}' if g else entry['grid']
        status = 'PASS' if result.returncode == 0 and mcells > 0 else 'FAIL'
        gpu_example_results.append({
            'file': fname, 'status': status,
            'time_s': round(elapsed,1),
            'mcells_s': mcells, 'grid': grid_str
        })
        print(f'{status} | {elapsed:.1f}s | {mcells:.1f} Mcells/s')
    except subprocess.TimeoutExpired:
        gpu_example_results.append({'file': fname, 'status': 'TIMEOUT', 'time_s': 300, 'mcells_s': 0, 'grid': ''})
        print('TIMEOUT')

print('\nAll examples done.')

In [ ]:
# ── 7. Save this session's results ────────────────────────────────────────────
import platform

gpu_name = torch.cuda.get_device_name(0).replace(' ', '_')
session_results = {
    'meta': {
        'date': __import__('datetime').datetime.now().isoformat(),
        'gpu': torch.cuda.get_device_name(0),
        'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'python': platform.python_version(),
    },
    'grid_scaling': gpu_scaling_results,
    'examples': gpu_example_results,
}

out_path = f'benchmarks/colab_{gpu_name}_results.json'
with open(out_path, 'w') as f:
    json.dump(session_results, f, indent=2)
print(f'Saved → {out_path}')

In [ ]:
# ── 8. Visualise: Grid Scaling — CPU vs GPU ───────────────────────────────────
import matplotlib
matplotlib.use('Agg')   # remove if running interactively
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('WaveForge 3D — Grid Scaling Benchmark', fontsize=14, fontweight='bold')

# ── Throughput (Mcells/s) ──
ax = axes[0]

# CPU (local dev)
cpu_N  = [r['N'] for r in cpu_scaling['waveforge_cpu']]
cpu_mc = [r['mcells_s'] for r in cpu_scaling['waveforge_cpu']]
ax.plot(cpu_N, cpu_mc, 'o--', color='royalblue', label='WaveForge CPU (dev)', lw=2)

# Kaggle T4
t4_N  = [r['N'] for r in kaggle['gpu_scaling']['cuda:0']['rows']]
t4_mc = [r['mcells_s'] for r in kaggle['gpu_scaling']['cuda:0']['rows']]
ax.plot(t4_N, t4_mc, 's-', color='darkorange', label='Kaggle T4', lw=2)

# This Colab GPU
col_N  = [r['N'] for r in gpu_scaling_results]
col_mc = [r['mcells_s'] for r in gpu_scaling_results]
ax.plot(col_N, col_mc, '^-', color='crimson', label=f'Colab {torch.cuda.get_device_name(0).split()[0]}', lw=2)

# PyMEEP CPU reference (2D, plotted as dashed for context)
meep_N  = [r['N'] for r in cpu_scaling['meep_cpu']]
meep_mc = [r['mcells_s'] for r in cpu_scaling['meep_cpu']]
ax.plot(meep_N, meep_mc, 'x:', color='gray', label='PyMEEP CPU (2D ref)', lw=1.5)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Grid size N (N³ cells)')
ax.set_ylabel('Throughput (Mcells/s)')
ax.set_title('Throughput vs Grid Size')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}³'))

# ── Speedup vs CPU ──
ax2 = axes[1]
# Interpolate GPU/CPU speedup at matching grid sizes
cpu_dict = {r['N']: r['mcells_s'] for r in cpu_scaling['waveforge_cpu']}
# Kaggle T4 speedup
t4_speedup_N  = [r['N'] for r in kaggle['gpu_scaling']['cuda:0']['rows'] if r['N'] in cpu_dict]
t4_speedup    = [kaggle['gpu_scaling']['cuda:0']['rows'][i]['mcells_s'] / cpu_dict[r['N']]
                 for i, r in enumerate(kaggle['gpu_scaling']['cuda:0']['rows']) if r['N'] in cpu_dict]
ax2.plot(t4_speedup_N, t4_speedup, 's-', color='darkorange', label='Kaggle T4 speedup', lw=2)

# Colab GPU speedup
col_speedup_N = [r['N'] for r in gpu_scaling_results if r['N'] in cpu_dict]
col_speedup   = [r['mcells_s'] / cpu_dict[r['N']] for r in gpu_scaling_results if r['N'] in cpu_dict]
ax2.plot(col_speedup_N, col_speedup, '^-', color='crimson',
         label=f'Colab {torch.cuda.get_device_name(0).split()[0]} speedup', lw=2)

ax2.axhline(y=1, color='gray', ls='--', lw=1)
ax2.set_xscale('log', base=2)
ax2.set_xlabel('Grid size N (N³ cells)')
ax2.set_ylabel('GPU / CPU speedup')
ax2.set_title('GPU Speedup over CPU')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}³'))

plt.tight_layout()
plt.savefig('docs/assets/colab_scaling_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/colab_scaling_benchmark.png')

In [ ]:
# ── 9. Visualise: All 10 Examples — CPU vs GPU ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('WaveForge 3D — All 10 Examples: CPU vs GPU', fontsize=14, fontweight='bold')

labels = [r['file'].replace('3d_', '').replace('.py', '') for r in cpu_3d]
cpu_mc_ex = [r['mcells_s'] for r in cpu_3d]
gpu_mc_ex = [r['mcells_s'] for r in gpu_example_results]

x = np.arange(len(labels))
width = 0.35

# ── Throughput bar chart ──
ax = axes[0]
bars_cpu = ax.bar(x - width/2, cpu_mc_ex, width, label='CPU', color='royalblue', alpha=0.85)
bars_gpu = ax.bar(x + width/2, gpu_mc_ex, width, label=f'GPU ({torch.cuda.get_device_name(0).split()[0]})',
                  color='crimson', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)')
ax.set_title('Throughput per Example')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
# Annotate speedup on top of GPU bars
for i, (c, g) in enumerate(zip(cpu_mc_ex, gpu_mc_ex)):
    if c > 0:
        ax.text(x[i] + width/2, g + 0.3, f'{g/c:.1f}x', ha='center', fontsize=7, color='darkred')

# ── Runtime bar chart ──
ax2 = axes[1]
cpu_time_ex = [r['time_s'] for r in cpu_3d]
gpu_time_ex = [r['time_s'] for r in gpu_example_results]
ax2.bar(x - width/2, cpu_time_ex, width, label='CPU', color='royalblue', alpha=0.85)
ax2.bar(x + width/2, gpu_time_ex, width, label=f'GPU ({torch.cuda.get_device_name(0).split()[0]})',
        color='crimson', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Wall time (s)')
ax2.set_title('Runtime per Example')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/colab_examples_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/colab_examples_benchmark.png')

In [ ]:
# ── 10. Visualise: PyMEEP vs WaveForge CPU vs WaveForge GPU ──────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('WaveForge vs PyMEEP — Throughput Comparison', fontsize=14, fontweight='bold')

scene_names = list(meep_cmp.keys())
meep_mc_vals   = [meep_cmp[s]['meep']['mcells_s'] for s in scene_names]
wf_cpu_mc_vals = [meep_cmp[s]['waveforge']['mcells_s'] for s in scene_names]

# WaveForge GPU: take average from GPU examples (coarse match by index)
wf_gpu_mc_vals = []
for i, s in enumerate(scene_names):
    if i < len(gpu_example_results):
        wf_gpu_mc_vals.append(gpu_example_results[i]['mcells_s'])
    else:
        wf_gpu_mc_vals.append(0)

x = np.arange(len(scene_names))
width = 0.25
ax.bar(x - width, meep_mc_vals,   width, label='PyMEEP CPU',    color='gray',      alpha=0.85)
ax.bar(x,         wf_cpu_mc_vals, width, label='WaveForge CPU', color='royalblue', alpha=0.85)
ax.bar(x + width, wf_gpu_mc_vals, width, label=f'WaveForge GPU ({torch.cuda.get_device_name(0).split()[0]})',
       color='crimson', alpha=0.85)

# Annotate GPU/Meep speedup
for i, (m, g) in enumerate(zip(meep_mc_vals, wf_gpu_mc_vals)):
    if m > 0 and g > 0:
        ax.text(x[i] + width, g + 0.5, f'{g/m:.0f}x\nvs\nMeep', ha='center', fontsize=7, color='darkred')

ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in scene_names], fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)')
ax.set_title('PyMEEP CPU vs WaveForge CPU vs WaveForge GPU')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/colab_meep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/colab_meep_comparison.png')

In [ ]:
# ── 11. Summary Table ─────────────────────────────────────────────────────────
print('=' * 70)
print(f'  WAVEFORGE 3D BENCHMARK SUMMARY — {torch.cuda.get_device_name(0)}')
print('=' * 70)

# Grid scaling table
print(f'\n  Grid Scaling (N³ free-space, {N_WARMUP} warmup + {N_STEPS} timed steps):')
print(f'  {"N":>6} {"WF-CPU":>12} {"Kaggle-T4":>12} {"Colab-GPU":>12} {"Speedup":>10}')
print(f'  {"-"*56}')
cpu_dict2 = {r['N']: r['mcells_s'] for r in cpu_scaling['waveforge_cpu']}
t4_dict   = {r['N']: r['mcells_s'] for r in kaggle['gpu_scaling']['cuda:0']['rows']}
col_dict  = {r['N']: r['mcells_s'] for r in gpu_scaling_results}
all_Ns    = sorted(set(cpu_dict2) | set(t4_dict) | set(col_dict))
for N in all_Ns:
    cpu_v = cpu_dict2.get(N, 0)
    t4_v  = t4_dict.get(N, 0)
    col_v = col_dict.get(N, 0)
    spd   = f'{col_v/cpu_v:.1f}x' if cpu_v > 0 else '-'
    print(f'  {N:6d}³ {cpu_v:10.1f} {t4_v:12.1f} {col_v:12.1f} {spd:>10}')

print(f'\n  All 10 Examples:')
print(f'  {"Example":<35} {"CPU (Mcells/s)":>15} {"GPU (Mcells/s)":>15} {"Speedup":>10}')
print(f'  {"-"*78}')
total_cpu = total_gpu = 0
for c, g in zip(cpu_3d, gpu_example_results):
    name = c['file'].replace('3d_', '').replace('.py', '')
    spd  = f'{g["mcells_s"]/c["mcells_s"]:.1f}x' if c['mcells_s'] > 0 else '-'
    print(f'  {name:<35} {c["mcells_s"]:>15.1f} {g["mcells_s"]:>15.1f} {spd:>10}')
    total_cpu += c['time_s']
    total_gpu += g['time_s']

print(f'  {"Total wall time":<35} {total_cpu:>13.0f}s {total_gpu:>13.0f}s {total_cpu/total_gpu:>9.1f}x')
print()

# Peak GPU
if col_dict:
    peak_gpu = max(col_dict.values())
    peak_N   = max(col_dict, key=lambda k: col_dict[k])
    print(f'  Peak GPU throughput: {peak_gpu:.1f} Mcells/s at {peak_N}³')

print('=' * 70)

In [ ]:
# ── 12. Download results to local machine ─────────────────────────────────────
try:
    from google.colab import files
    files.download(out_path)
    print(f'Downloaded: {out_path}')
except ImportError:
    print('Not on Colab — results saved to', out_path)